# Week 5 · Sunday quiz

**Rules.** Part A: write the exact output of each snippet on paper, then run it. Part B: on paper, then open the answer. Part C: write each function on paper, then type it and run the check. About 40 minutes.

Log all three scores, and for each miss write **which idea** it was (e.g. "A6: gradients accumulate").

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import checks; checks.load(globals(), "08")      # defines check(n)

## Part A · Predict the output

**A1**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
print(torch.tensor([1, 2]).dtype, torch.tensor([1.0, 2.0]).dtype)

**A2**

In [ ]:
print(nn.Linear(5, 3).weight.shape, nn.Linear(5, 3).bias.shape)

**A3**

In [ ]:
x = torch.ones(2, 3)
print(x.sum(dim=1).shape, x.sum(dim=1, keepdim=True).shape)

**A4**

In [ ]:
print(round(nn.CrossEntropyLoss()(torch.zeros(1, 4), torch.tensor([2])).item(), 4))

**A5**

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
(w ** 3).backward()
print(w.grad)

**A6**

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for _ in range(2):
    (5 * w).backward()
print(w.grad)

**A7**

In [ ]:
import math
print(torch.softmax(torch.tensor([0.0, 0.0, math.log(2)]), dim=0))

**A8**

In [ ]:
print(sum(p.numel() for p in nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2)).parameters()))

**A9**

In [ ]:
X = np.arange(6).reshape(2, 3); W = np.ones((3, 2))
print((X @ W).shape, (X[:, :, None] * W[None]).sum(axis=1).tolist())

**A10**

In [ ]:
drop = nn.Dropout(0.5); drop.eval()
print(drop(torch.ones(3)))

## Part B · On paper

**B1.** One sigmoid neuron with binary cross-entropy: $x = (1, 2)$, $w = (0.5, -0.25)$, $b = 0$, label $y = 1$. Compute $p$, $\partial L / \partial w$ and $\partial L / \partial b$.

<details><summary>Answer</summary>

$z = 0.5 - 0.5 + 0 = 0$, so $p = \sigma(0) = 0.5$. $\partial L / \partial z = p - y = -0.5$, so $\partial L / \partial w = -0.5 \cdot (1, 2) = (-0.5, -1)$ and $\partial L / \partial b = -0.5$. Gradient descent will raise both weights and the bias, pushing $p$ up towards 1.

</details>

**B2.** Adam, first step ($t = 1$), $\eta = 0.01$, gradient $g = 250$. By how much does the weight change? Would it be different for $g = 0.002$?

<details><summary>Answer</summary>

$\hat m_1 = g$ and $\hat s_1 = g^2$ after bias correction, so the step is $\eta \cdot g / \lvert g \rvert = 0.01$: the weight decreases by $0.01$. The same for $g = 0.002$: Adam's first step has size $\eta$ whatever the gradient's scale.

</details>

**B3.** An MLP $784 \to 256 \to 128 \to 10$ (ReLU between). How many parameters? For a batch of 64 images, what are the shapes of the input and of the output?

<details><summary>Answer</summary>

$784 \cdot 256 + 256 = 200{,}960$; $256 \cdot 128 + 128 = 32{,}896$; $128 \cdot 10 + 10 = 1290$; total **235,146**. Input `(64, 784)` (after flattening `(64, 1, 28, 28)`), output `(64, 10)` logits.

</details>

## Part C · Checked functions

Write each function on paper first, then type it and run the check.

**C1 · Predict with a finished MLP.** `Ws`, `bs`: a trained network's layers (weights `(in, out)`, biases `(out,)`). ReLU between layers, logits at the end. Return the predicted class of each row of `X`, **shape `(n,)`**.

**Example**: what goes in, and what should come out:
```python
Ws = [np.array([[1.0, -1.0]]), np.eye(2)]; bs = [np.zeros(2), np.zeros(2)]     # 1 input -> 2 ReLU units -> 2 classes
mlp_predict(np.array([[2.0], [-3.0]]), Ws, bs)    # → [0 1]   hidden (2, 0) -> class 0;  hidden (0, 3) -> class 1
```

In [ ]:
def mlp_predict(X, Ws, bs):
    pass  # your code

In [ ]:
check(1)

**C2 · Run momentum.** Starting from `w0`, take `steps` momentum steps ($v \leftarrow \beta v + \nabla f(w)$, $w \leftarrow w - \eta v$, $v$ starting at 0) and return the final `w`. Don't change `w0`.

**Example**: what goes in, and what should come out:
```python
momentum_run(lambda w: 2 * w, np.array([1.0]), 0.1, 0.9, 3)    # → [0.062]   05 §3's drill
```

In [ ]:
def momentum_run(grad, w0, lr, beta, steps):
    pass  # your code

In [ ]:
check(2)

**C3 · Adam, from memory.** One Adam step at step number `t`: update `m` and `s`, bias-correct them, update `w`. Return `(w_new, m_new, s_new)`.

**Example**: what goes in, and what should come out:
```python
adam_step(np.array([1.0]), np.zeros(1), np.zeros(1), np.array([2.0]), 1, lr=0.1)[0]    # → [0.9]
```

In [ ]:
def adam_step(w, m, s, g, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    pass  # your code

In [ ]:
check(3)

**C4 · Predict with a PyTorch model.** `X` is a NumPy array `(n, d)`. Put the model in evaluation mode, turn off gradient recording, and return the predicted classes as a **NumPy** int array, **shape `(n,)`**. (The model in the check has dropout, so evaluation mode matters.)

**Example**: what goes in, and what should come out:
```python
torch_predict(nn.Sequential(nn.Linear(2, 3)), np.zeros((4, 2)))    # → an int64 array of shape (4,)
```

In [ ]:
def torch_predict(model, X):
    pass  # your code

In [ ]:
check(4)

**C5 · Count the trainable parameters.** Return the number of parameter entries that will be trained, i.e. those with `requires_grad=True` (frozen layers don't count; you'll freeze layers in transfer learning).

**Example**: what goes in, and what should come out:
```python
count_trainable(nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 2)))    # → 23
```

In [ ]:
def count_trainable(model):
    pass  # your code

In [ ]:
check(5)

---
Log all three scores (A, B, C) and every miss **by topic**, in the tracker. Week 6's recaps are built from them.